# EXPLORATION - NOT MAIN PORTFOLIO
# DATE: 3/3/26

# FINDINGS TO BE ADDED TO MAIN:
- baseline already gave significant performance
- SMOTE oversampling resulted in higher recall but slightly lower precision and f1 score than baseline
- The tradeoff are justifiable, the final model will use SMOTE oversample

## Goal

Performing oversampling and undersampling because the data is imbalanced as concluded from the initial exploration. Testing several sampling techniques using baseline model for the performance comparison. Final outcome is whether we use the baseline dataset or one after sampling

## Setup

In [5]:
import pandas as pd

In [6]:
data = pd.read_csv('creditcard.csv')

data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


as mention before, no engineering will be done currently thus we use the raw data

In [23]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score \
, f1_score

#for imbalance handling
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek, SMOTEENN

for this one we also use Random Forest for performance testing

In [8]:
X = data.drop('Class', axis=1)
y = data['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

print(f'Training set: {len(X_train)} samples')
print(f'Test set: {len(X_test)} samples')
print(f'Training fraud rate: {y_train.mean()*100:.3f} samples')
print(f'Training fraud rate: {y_test.mean()*100:.3f} samples')

Training set: 227845 samples
Test set: 56962 samples
Training fraud rate: 0.173 samples
Training fraud rate: 0.172 samples


## Imbalance Handling
### Baseline Performance

In [9]:
model = RandomForestClassifier(n_estimators=100, random_state=1, n_jobs=-1)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Baseline: No Handling")
print(classification_report(y_test, y_pred))

baseline_metrics = {
    'precision' : precision_score(y_test, y_pred),
    'recall' : recall_score(y_test, y_pred),
    'f1' : f1_score(y_test, y_pred)
}

#confusion matrix
cm_baseline = confusion_matrix(y_test, y_pred)
print("Confusion Matrix")
print(cm_baseline)

Baseline: No Handling
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.93      0.85      0.89        98

    accuracy                           1.00     56962
   macro avg       0.97      0.92      0.94     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix
[[56858     6]
 [   15    83]]


Pretty good recall 85% with 15 fraud being misclassified as legitimate

### Random Undersampling

In [10]:
#remember to only sample training data
under_sample = RandomUnderSampler(random_state=1)
X_train_under, y_train_under = under_sample.fit_resample(X_train, y_train)

print(f'Undersample training set: {len(X_train_under)} samples')
print(f'Undersample fraud rate: {y_train_under.mean()*100:.2f} samples')

#use the same model
model.fit(X_train_under, y_train_under)
y_pred_under = model.predict(X_test)

print("Random Undersampling")
print(classification_report(y_test, y_pred_under))

under_metrics = {
    'precision' : precision_score(y_test, y_pred_under),
    'recall' : recall_score(y_test, y_pred_under),
    'f1' : f1_score(y_test, y_pred_under)
}

#confusion matrix
cm_under = confusion_matrix(y_test, y_pred_under)
print("Confusion Matrix")
print(cm_under)

Undersample training set: 788 samples
Undersample fraud rate: 50.00 samples
Random Undersampling
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.07      0.92      0.12        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.56     56962
weighted avg       1.00      0.98      0.99     56962

Confusion Matrix
[[55593  1271]
 [    8    90]]


random undersample reduced the data to only have 788 samples, with the fraud rate being 50% of the undersampled data. The recall rate improve to 92% but the precision dropped to 7% and f1 score dropped to 12%

However, even though we have less false negative, the false positive increased significantly which is also not a good idea

### SMOTE Oversampling

In [11]:
smote_sample = SMOTE(random_state=1)
X_train_smote, y_train_smote = smote_sample.fit_resample(X_train, y_train)

print(f'Undersample training set: {len(X_train_smote)} samples')
print(f'Undersample fraud rate: {y_train_smote.mean()*100:.2f} samples')

#use the same model
model.fit(X_train_smote, y_train_smote)
y_pred_smote = model.predict(X_test)

print("SMOTE")
print(classification_report(y_test, y_pred_smote))

smote_metrics = {
    'precision' : precision_score(y_test, y_pred_smote),
    'recall' : recall_score(y_test, y_pred_smote),
    'f1' : f1_score(y_test, y_pred_smote)
}

#confusion matrix
cm_smote = confusion_matrix(y_test, y_pred_smote)
print("Confusion Matrix")
print(cm_smote)

Undersample training set: 454902 samples
Undersample fraud rate: 50.00 samples
SMOTE
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.87      0.88      0.87        98

    accuracy                           1.00     56962
   macro avg       0.93      0.94      0.94     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix
[[56851    13]
 [   12    86]]


smote oversample increased the data so the fraud rate is 50%. The recall rate also improved to 88% but the precision dropped slightly to 87%

unlike random downsampling, SMOTE improved the recall without hurting the precision & f1 score, the false positive is kept at low

### SMOTE + TOMEK

In [12]:
smote_tomek = SMOTETomek(random_state=1)
X_train_st, y_train_st = smote_tomek.fit_resample(X_train, y_train)

print(f'Undersample training set: {len(X_train_st)} samples')
print(f'Undersample fraud rate: {y_train_st.mean()*100:.2f} samples')

#use the same model
model.fit(X_train_st, y_train_st)
y_pred_st = model.predict(X_test)

print("SMOTE TOMEK")
print(classification_report(y_test, y_pred_st))

st_metrics = {
    'precision' : precision_score(y_test, y_pred_st),
    'recall' : recall_score(y_test, y_pred_st),
    'f1' : f1_score(y_test, y_pred_st)
}

#confusion matrix
cm_st = confusion_matrix(y_test, y_pred_st)
print("Confusion Matrix")
print(cm_st)

Undersample training set: 453652 samples
Undersample fraud rate: 50.00 samples
SMOTE TOMEK
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.87      0.88      0.87        98

    accuracy                           1.00     56962
   macro avg       0.93      0.94      0.94     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix
[[56851    13]
 [   12    86]]


SMOTE Tomek gave the same result as SMOTE, with recall score 86% and f1 score 87%

### SMOTE + ENN

In [20]:
smote_enn = SMOTEENN(random_state=1)
X_train_se, y_train_se = smote_enn.fit_resample(X_train, y_train)

print(f'Undersample training set: {len(X_train_se)} samples')
print(f'Undersample fraud rate: {y_train_se.mean()*100:.2f} samples')

#use the same model
model.fit(X_train_se, y_train_se)
y_pred_se = model.predict(X_test)

print("SMOTE ENN")
print(classification_report(y_test, y_pred_se))

se_metrics = {
    'precision' : precision_score(y_test, y_pred_se),
    'recall' : recall_score(y_test, y_pred_se),
    'f1' : f1_score(y_test, y_pred_se)
}

#confusion matrix
cm_se = confusion_matrix(y_test, y_pred_se)
print("Confusion Matrix")
print(cm_se)

Undersample training set: 427872 samples
Undersample fraud rate: 51.07 samples
SMOTE ENN
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.83      0.88      0.86        98

    accuracy                           1.00     56962
   macro avg       0.92      0.94      0.93     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix
[[56847    17]
 [   12    86]]


SMOTE ENN also gave the same recall score as SMOTE at 88%, but it does have lower precision of 83% and f1 score of 86%

### Class Weights

In [21]:
weights_model = RandomForestClassifier(n_estimators=100, random_state=1, class_weight='balanced'
                                      , n_jobs=-1)

weights_model.fit(X_train, y_train)
y_pred_w = weights_model.predict(X_test)

print("Class Weights")
print(classification_report(y_test, y_pred_w))

cw_metrics = {
    'precision' : precision_score(y_test, y_pred_w),
    'recall' : recall_score(y_test, y_pred_w),
    'f1' : f1_score(y_test, y_pred_w)
}

#confusion matrix
cm_w = confusion_matrix(y_test, y_pred_w)
print("Confusion Matrix")
print(cm_w)

Class Weights
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.92      0.85      0.88        98

    accuracy                           1.00     56962
   macro avg       0.96      0.92      0.94     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix
[[56857     7]
 [   15    83]]


Adding class weight metric to the model gave the same recall score as baseline at 85% but reduced the precision & f1 score slightly to 92% and 88% respectively

## Decision
### Summary

In [22]:
# Create comparison table
results_df = pd.DataFrame({
    'Experiment': ['Baseline', 'RandomUnder', 'SMOTE', 'SMOTE Tomek', 'SMOTE ENN', 'Class Weights'],
    'Precision': [baseline_metrics['precision'], under_metrics['precision'], 
                  smote_metrics['precision'], st_metrics['precision'],
                  se_metrics['precision'], cw_metrics['precision']],
    'Recall': [baseline_metrics['recall'], under_metrics['recall'], 
               smote_metrics['recall'], st_metrics['recall'],
               se_metrics['recall'], cw_metrics['recall']],
    'F1': [baseline_metrics['f1'], under_metrics['f1'], 
           smote_metrics['f1'], st_metrics['f1'],
           se_metrics['f1'], cw_metrics['f1']]
})

print(results_df)

      Experiment  Precision    Recall        F1
0       Baseline   0.932584  0.846939  0.887701
1    RandomUnder   0.066128  0.918367  0.123372
2          SMOTE   0.868687  0.877551  0.873096
3    SMOTE Tomek   0.868687  0.877551  0.873096
4      SMOTE ENN   0.834951  0.877551  0.855721
5  Class Weights   0.922222  0.846939  0.882979


Analysis:
- The baseline performance are actually already pretty good with recall score 85%, precision 93%, and f1 score 89%
- Random Undersampling have the highest recall of 90% but the rest is miserable with precision 7% and f1 score 12%
- SMOTE gave a better improvement with recall score 88% but have slight trade off of precision 87% and f1 score 87%
- SMOTE + Tomek have the same result as SMOTE and SMOTE + ENN have the same recall as SMOTE but with lower precision & f1 score
- Class weights have the same recall as baseline but with lower precision and f1 score

### Final Decision

While baseline performance is already good, I am going to select SMOTE oversampling because it improved recall from 84.7% to 87.7%. Even though the precision dropped from 93% to 87% and f1 score from 89% to 87%, it is still a high score. In addition, I believe that in fraud detection, the cost of missing a fraud (false negative) typically outweights the cost of false alarms (false positive). My opinion is that the trade-off are justifiable.

The final model will be using SMOTE oversampled data